# SportsStats Sport Analysis

## Project Overview

SportsStats is a sports analysis firm partnering with local news and elite personal trainers to provide “interesting” insights to help their partners. This project analyzes 120 years of Olympic athlete data to deliver actionable insights for SportsStats and its dual audience of local news media and elite personal trainers. By examining physical attributes alongside historical medal performances, the study uncovers optimal physical profiles for specific sports as well as compelling narrative trends across competing nations. Elite trainers can leverage these biometric benchmarks to tailor training programs for aspiring athletes, while news outlets can highlight rising sporting nations and historical anomalies. The findings bridge raw sports statistics with practical health coaching and media storytelling. Ultimately, this analysis demonstrates how long-term sports data can inform both human performance optimization and public journalism.


## Data Import

In [ ]:
!pip uninstall -y jupysql ipython-sql sql
!pip install "ipython-sql==0.4.1" "sqlalchemy<1.4.0" "prettytable<3.0.0"

In [ ]:
import os
print(os.listdir())

In [1]:
import pandas as pd

df = pd.read_csv('athlete_events.csv')

# View the first 5 rows
df.head()

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN


In [2]:
df.columns

Index(['ID', 'Name', 'Sex', 'Age', 'Height', 'Weight', 'Team', 'NOC', 'Games',
       'Year', 'Season', 'City', 'Sport', 'Event', 'Medal'],
      dtype='str')

## Questions we want to answer

Biometrics / Personal Trainer Focus: How do physical attributes (Height, Weight, Age, BMI) vary by sport and position, and have these optimal athletic profiles evolved over time?

Media / News Story Focus: Which non-dominant nations (NOCs) show the highest rate of medal growth over the last 30 years, and does athlete delegation size directly correlate with medal count?

Gender & Aging Focus: How has the age distribution of female medalists evolved across Summer vs. Winter Games over the 120-year span?

## Hypothesis
Hypothesis 1 (Biometrics): Athletes in power-based sports (e.g., Weightlifting, Shot Put) have significantly higher BMI and lower height-to-weight ratios compared to endurance sports (e.g., Marathon, Cycling), and these physical ideal profiles have become more specialized over time.

Hypothesis 2 (National Performance): A country’s total medal yield is strongly positively correlated with the total number of athletes sent, but certain smaller nations achieve a much higher "medal-per-athlete" efficiency ratio.

Hypothesis 3 (Age Dynamics): The average age of Olympic medalists has steadily increased in technical sports while remaining under 23 in agility/gymnastics sports.

## Approach 
To answer these questions, I will first engineer new feature columns such as `BMI` and `Is_Medalist`. Next, I will evaluate the biometric relationships across different Sport categories to map physical profiles for personal trainers. For national trends, I will aggregate total athletes and medal counts by NOC and Year to calculate medal efficiency ratios (Medals / Athletes). Finally, I will analyze time-series trends by filtering for post-WWII or modern games (Year >= 1960) to eliminate historical gaps. These insights will be measured using correlation coefficients, average biometric spreads, and ranking distributions.

## Data cleaning

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 271116 entries, 0 to 271115
Data columns (total 15 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   ID      271116 non-null  int64  
 1   Name    271116 non-null  str    
 2   Sex     271116 non-null  str    
 3   Age     261642 non-null  float64
 4   Height  210945 non-null  float64
 5   Weight  208241 non-null  float64
 6   Team    271116 non-null  str    
 7   NOC     271116 non-null  str    
 8   Games   271116 non-null  str    
 9   Year    271116 non-null  int64  
 10  Season  271116 non-null  str    
 11  City    271116 non-null  str    
 12  Sport   271116 non-null  str    
 13  Event   271116 non-null  str    
 14  Medal   39783 non-null   str    
dtypes: float64(3), int64(2), str(10)
memory usage: 31.0 MB


In [4]:
import numpy as np
import sqlite3

In [5]:
# Clean Missing Values
# Fill NaN in 'Medal' with 'No Medal' (since NaN simply means the athlete didn't win a medal)
df['Medal'] = df['Medal'].fillna('No Medal')

# Add Feature Engineering (BMI)
# Convert Height (cm) to meters for BMI calculation: Weight (kg) / Height (m)^2
df['BMI'] = np.where(
    df['Height'].notnull() & df['Weight'].notnull(),
    df['Weight'] / ((df['Height'] / 100) ** 2),
    np.nan,
)

# Standardize Data Types
df['Age'] = df['Age'].astype('Int64')  # Nullable integer type
df['Year'] = df['Year'].astype(int)

# Export Clean Data to SQLite for jupysql
conn = sqlite3.connect('olympics.db')
df.to_sql('olympics', conn, if_exists='replace', index=False)
conn.close()

print('Data cleaned and successfully loaded into sqlite database "olympics.db"')

Data cleaned and successfully loaded into sqlite database "olympics.db"


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 271116 entries, 0 to 271115
Data columns (total 16 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   ID      271116 non-null  int64  
 1   Name    271116 non-null  str    
 2   Sex     271116 non-null  str    
 3   Age     261642 non-null  Int64  
 4   Height  210945 non-null  float64
 5   Weight  208241 non-null  float64
 6   Team    271116 non-null  str    
 7   NOC     271116 non-null  str    
 8   Games   271116 non-null  str    
 9   Year    271116 non-null  int64  
 10  Season  271116 non-null  str    
 11  City    271116 non-null  str    
 12  Sport   271116 non-null  str    
 13  Event   271116 non-null  str    
 14  Medal   271116 non-null  str    
 15  BMI     206853 non-null  float64
dtypes: Int64(1), float64(3), int64(2), str(10)
memory usage: 33.4 MB


In [8]:
import sqlite3
import pandas as pd

def run_sql(query):
    conn = sqlite3.connect('olympics.db')
    df_result = pd.read_sql_query(query, conn)
    conn.close()
    return df_result

In [9]:
query = '''SELECT 
    COUNT(*) AS total_records,
    COUNT(DISTINCT ID) AS unique_athletes,
    COUNT(DISTINCT NOC) AS total_countries,
    COUNT(DISTINCT Sport) AS total_sports,
    SUM(CASE WHEN Age IS NULL THEN 1 ELSE 0 END) AS missing_age_count,
    SUM(CASE WHEN Height IS NULL THEN 1 ELSE 0 END) AS missing_height_count,
    SUM(CASE WHEN Weight IS NULL THEN 1 ELSE 0 END) AS missing_weight_count,
    SUM(CASE WHEN Medal != 'No Medal' THEN 1 ELSE 0 END) AS total_medals_awarded
FROM olympics;'''
run_sql(query)

,total_records,unique_athletes,total_countries,total_sports,missing_age_count,missing_height_count,missing_weight_count,total_medals_awarded
0,271116,135571,230,66,9474,60171,62875,39783


In [10]:
query = '''
SELECT 
    CASE 
        WHEN Year < 1920 THEN '1896-1919'
        WHEN Year < 1960 THEN '1920-1959'
        WHEN Year < 2000 THEN '1960-1999'
        ELSE '2000-Present'
    END AS Era,
    Season,
    COUNT(DISTINCT ID) AS Female_Medalists,
    ROUND(AVG(Age), 1) AS Avg_Age,
    MIN(Age) AS Min_Age,
    MAX(Age) AS Max_Age
FROM olympics
WHERE Sex = 'F' 
  AND Medal != 'No Medal' 
  AND Age IS NOT NULL
GROUP BY Era, Season
ORDER BY Era, Season;'''
run_sql(query)

,Era,Season,Female_Medalists,Avg_Age,Min_Age,Max_Age
0,1896-1919,Summer,52,29.7,15,63
1,1920-1959,Summer,495,22.8,11,69
2,1920-1959,Winter,63,23.3,15,39
3,1960-1999,Summer,2882,23.2,13,52
4,1960-1999,Winter,459,24.4,13,42
5,2000-Present,Summer,3251,25.6,15,52
6,2000-Present,Winter,594,26.0,15,46


In [14]:
query = '''
SELECT 
    CASE 
        WHEN Year < 1920 THEN '1896-1919'
        WHEN Year < 1960 THEN '1920-1959'
        WHEN Year < 2000 THEN '1960-1999'
        ELSE '2000-Present'
    END AS Era,
    Season,
    Sex,
    COUNT(DISTINCT ID) AS Medalists,
    ROUND(AVG(Age), 1) AS Avg_Age,
    MIN(Age) AS Min_Age,
    MAX(Age) AS Max_Age
FROM olympics
WHERE Medal != 'No Medal' 
  AND Age IS NOT NULL
GROUP BY Era, Season, Sex
ORDER BY Era, Season, Sex;'''

run_sql(query)

,Era,Season,Sex,Medalists,Avg_Age,Min_Age,Max_Age
0,1896-1919,Summer,F,52,29.7,15,63
1,1896-1919,Summer,M,2163,27.3,10,71
2,1920-1959,Summer,F,495,22.8,11,69
3,1920-1959,Summer,M,4761,27.8,12,73
4,1920-1959,Winter,F,63,23.3,15,39
5,1920-1959,Winter,M,619,27.2,15,58
6,1960-1999,Summer,F,2882,23.2,13,52
7,1960-1999,Summer,M,7151,25.4,13,66
8,1960-1999,Winter,F,459,24.4,13,42
9,1960-1999,Winter,M,1323,25.5,14,50


In [12]:
query = '''SELECT 
    Sport,
    Sex,
    ROUND(AVG(Age), 1) AS Avg_Age,
    ROUND(AVG(Height), 1) AS Avg_Height_cm,
    ROUND(AVG(Weight), 1) AS Avg_Weight_kg,
    ROUND(AVG(BMI), 1) AS Avg_BMI
FROM olympics
WHERE Sport IN ('Gymnastics', 'Weightlifting', 'Athletics', 'Swimming', 'Equestrianism')
  AND Height IS NOT NULL 
  AND Weight IS NOT NULL
GROUP BY Sport, Sex
ORDER BY Sport, Sex;'''
run_sql(query)

,Sport,Sex,Avg_Age,Avg_Height_cm,Avg_Weight_kg,Avg_BMI
0,Athletics,F,25.1,169.3,60.2,20.9
1,Athletics,M,25.4,179.8,73.8,22.7
2,Equestrianism,F,34.2,167.9,58.6,20.8
3,Equestrianism,M,34.1,176.6,70.9,22.7
4,Gymnastics,F,18.7,156.2,47.8,19.5
5,Gymnastics,M,23.8,167.6,63.3,22.5
6,Swimming,F,19.6,171.5,61.5,20.9
7,Swimming,M,21.5,184.4,78.0,22.9
8,Weightlifting,F,24.0,160.5,67.8,26.0
9,Weightlifting,M,25.5,169.2,81.7,28.0


## Proposed ERD

In [ ]:
from typing import Optional
from pydantic import BaseModel
import erdantic as erd

class Athletes(BaseModel):
    athlete_id: int
    name: str
    sex: str
    height: Optional[float]
    weight: Optional[float]
    bmi: Optional[float]

class Games(BaseModel):
    game_id: int
    games: str
    year: int
    season: str
    city: str

class NOCs(BaseModel):
    noc_code: str
    team_name: str

class FactOlympicResults(BaseModel):
    result_id: int
    sport: str
    event: str
    age_at_game: Optional[int]
    medal: str
    
    # Clean foreign key + model references
    athlete_id: Athletes
    game_id: Games
    noc_code: NOCs

# Create and save diagram
diagram = erd.create(FactOlympicResults)
diagram.draw("olympic_erd.png")

In [ ]:
diagram